# Gold Coast Beach Protection: Multi-Criteria Decision Analysis (MCDA)

This notebook presents a worked example of a Multi-Criteria Decision Analysis (MCDA) using **Preference Function Modeling (PFM)**. It evaluates alternative solutions to the Gold Coast beach erosion problem and demonstrates how stakeholders can assess design alternatives based on multiple criteria. These individual assessments are subsequently aggregated into an overall preference score for each alternative.

Many decision-making methods used in practice have a flawed mathematical foundation. These shortcomings often arise from the improper treatment of measurement scales, such as directly combining quantities expressed in different units (e.g., euros, kilograms, and square meters), aggregating ordinal data (e.g., 1st, 2nd, 3rd) with cardinal data (e.g., €1 million, €20 million, or 50 cm), or applying mathematical operations to scales for which they are not defined. For example, ranking aesthetics on a scale from 1 (best) to 4 (worst) does not imply that an alternative ranked 2 is twice as aesthetic as one ranked 4. Such practices implicitly assume measurement properties that the underlying data do not possess and can therefore lead to mathematically inconsistent results.

These issues are common in Multi-Criteria Decision Analysis (MCDA), where criteria often consist of a mixture of quantitative measures (e.g., costs or environmental impacts) and qualitative assessments (e.g., aesthetics or social acceptance). Arithmetic operations such as addition, averaging, and weighted summation are only meaningful when the underlying scales satisfy the required measurement properties. Applying these operations to ordinal scales can lead to mathematically inconsistent results, as different but equally valid numerical representations of the same preference ordering may produce different outcomes.

Preference Function Modeling addresses these challenges by first transforming each criterion onto a common preference scale prior to aggregation. This ensures that subsequent mathematical operations are meaningful, invariant to admissible transformations of the original data, and theoretically sound (Barzilai, 2010). This is achieved by eliciting preferences for each criterion and defining fixed reference points on the preference scale so that differences between alternatives become meaningful. A common approach, and the one used in this example, is to assign a value of 0 to the least preferred outcome and 100 to the most preferred outcome. The specific numerical values are not inherently important and may be changed (e.g., scales ranging from 50 to 150 are also used in the literature). What is essential is that the best and worst outcomes are explicitly defined and that intermediate alternatives are positioned proportionally between these fixed points.

Once stakeholder ratings have been correctly defined for each alternative, they must be aggregated into a single group preference score to support decision-making. PFM performs this aggregation using **affine aggregation**, which preserves the mathematical properties of the preference scales throughout the aggregation process.

This notebook demonstrates how to:

- Load and validate stakeholder ratings and criterion weights.
- Aggregate criterion ratings into a single preference score for each stakeholder (Level 1).
- Aggregate stakeholder preference scores into a single group preference score for each alternative (Level 2).

## Usage Guide

To perform an MCDA using PFM for your own System of Interest (SoI), you only need to modify the input **.csv** file containing the alternatives, criteria, stakeholder ratings, and criterion weights relevant to your case. A **.csv** (comma-separated values) file is a simple spreadsheet format that can be edited using software such as Microsoft Excel, LibreOffice Calc, or Visual Studio Code.

Provided that the structure of the input file is maintained, this notebook will automatically compute the aggregated preference scores for all stakeholders and the resulting group preference scores. Consequently, little to no modification of the notebook itself is required.

# Problem

During the 1990s, the city of Gold Coast experienced significant beach erosion, threatening one of its most valuable assets. The local economy relied heavily on beach-related recreation and surfing tourism, making the loss of beach width a major concern for the city council.

Four coastal protection alternatives were considered: **artificial seaweed**, an **artificial reef**, a **breakwater**, and **groynes**. For this analysis, five stakeholder groups — the Gold Coast City Council, the Local Community, the Research Community, the Surfing Community, and the Tourist Sector — each rated the alternatives against their own criteria, and assigned a weight to each criterion reflecting its relative importance to them.

This notebook combines those ratings into a single group preference score per alternative, so the alternatives can be ranked and compared.


## Importing Required Packages

Below, the required packages are imported. `numpy` and `pandas` are used for data handling and numerical operations. These packages are part of the `mude-base` environment that you created in week one of MUDE.  

The local module `a_fine_aggregator` performs the actual aggregation (see `a_fine_aggregator.py` for the implementation). It is imported using a **relative import**, which depends on the directory structure of the project. When the notebook is executed within the provided directory structure, this will work as intended. However, running the notebook outside of this directory will result in the following error: `ModuleNotFoundError: No module named 'genetic_algorithm_pfm'`


In [69]:
from pathlib import Path

print(Path.cwd())
print(list(Path.cwd().iterdir()))

c:\Users\tvanr\OneDrive\Documenten\BASE\Structural-group-5
[WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/.git'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/Civil-Engineering-Systems-Design-main'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/Drawing1.vsdx'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/EXAMPLE_Gold_Coast_MCDA.ipynb'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/genetic_algorithm_pfm'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/MCDA_Ratings_Werkspoor.csv'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/MCDA_Ratings_Werkspoor.xlsx'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/MCDA_Werkspoor.ipynb'), WindowsPath('c:/Users/tvanr/OneDrive/Documenten/BASE/Structural-group-5/~$MCDA_Ratings_Werkspoor.xlsx')]


In [70]:
# Import packages
import numpy as np
import pandas as pd

# Round the float values in the dataframe to 2 decimal places
pd.options.display.float_format = '{:.2f}'.format

# Import local module for a-fine-
import sys
from pathlib import Path

# Walk up to the folder that contains genetic_algorithm_pfm, regardless of where the notebook is run from
project_root = Path.cwd()
while not (project_root / "genetic_algorithm_pfm").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.append(str(project_root))

from genetic_algorithm_pfm.a_fine_aggregator import a_fine_aggregator


## Loading the Ratings Data

The stakeholder ratings are stored in `MCDA_Ratings_Gold_Coast.csv`. Each row represents the evaluation of all four design alternatives by a single stakeholder for one criterion, together with the weight (`Criteria_Weight`) that the stakeholder assigns to that criterion.

For improved readability, the same information is also presented in the figure below. The figure shows that each stakeholder evaluates the alternatives based on a different set of criteria, each with its own relative importance. Note that stakeholders can have different preference functions for the same criterion. For example, the City Council places a higher preference on altertives that trap more sand (erosion control) because it results in a wider beach, whereas the researchers are more interested in innovative erosion-control methods, such as artificial seaweed and artificial reefs.

The list of `alternatives` is extracted from the columns between `Stakeholder`/`Criteria` and `Criteria_Weight`, while the list of `stakeholders` is obtained from the unique values in the `Stakeholder` column. Both lists are printed, and the raw ratings table is displayed to verify that the data have been imported correctly.

<img src="MCDA_input/Ratings_Gold_Coast_alternatives.png" alt="Stakeholder ratings for the Gold Coast alternatives" width="800">

In [71]:
ratings = pd.read_csv(
    "MCDA_Ratings_Werkspoor.csv",  # path to the CSV file
    sep=";",       # fields are semicolon-separated, not comma-separated
    skiprows=0     # skip the "MCDA Gold Coast Design Alternatives" title row
)
ratings["Criteria_weight"] = (
    ratings["Criteria_weight"]
    .astype(str)
    .str.replace(",", ".", regex=False)   # handle Dutch decimal commas if present
    .astype(float)
)
alternatives = list(ratings.columns[2:-1].unique()) # Get the list of alternatives from the dataframe columns, excluding the first two and last column
stakeholders = list(ratings["Stakeholder"].unique()) # Get the list of stakeholders from the "Stakeholder" column in the dataframe
display(ratings)
print(alternatives)
print(stakeholders)


,Stakeholder,Criteria,Tunnel,Don't widen river,Truss bridge,Bridge somewhere else,Werkspoorbridge,Criteria_weight
0,Nature,Initial cost,2,10,5,5,5,0.05
1,Nature,Durability,9,5,8,8,8,0.10
2,Nature,Fit within surrounding (In terms of vegetation),6,10,8,5,8,0.10
3,Nature,Climate adaptation,10,3,8,8,8,0.20
4,Nature,Emissions during construction,2,10,5,5,7,0.25
5,Nature,Energy use during lifecycle,2,10,8,8,8,0.20
6,Nature,Aquatic life,9,10,5,5,5,0.10
7,Companies,Safety,8,4,7,7,7,0.20
8,Companies,Capacity,8,3,8,8,8,0.40
9,Companies,Durability,8,2,8,9,8,0.30


['Tunnel', "Don't widen river", 'Truss bridge', 'Bridge somewhere else', 'Werkspoorbridge']
['Nature', 'Companies', 'Government', 'Citizents']


## Checking the Criteria Weights

For the a-fine aggregator to produce a meaningful result, each stakeholder's criteria weights must sum to one (100%). Before aggregating anything, it is good practice to check this explicitly rather than assume the input data is correct — a silent mismatch here would otherwise propagate into the final scores without any warning.

The loop below sums the `Criteria_Weight` values per stakeholder and flags any stakeholder whose weights do not sum to 1 (within a small numerical tolerance) as a `MISMATCH`.


In [72]:
# Check each stakeholder's weights sum to 1 (i.e. 100%)
print("Check weights per stakeholder:")
all_valid = True

for stakeholder in stakeholders:
    stakeholder_weights = ratings.loc[ratings["Stakeholder"] == stakeholder, "Criteria_weight"]
    total = stakeholder_weights.sum()
    is_valid = np.isclose(total, 1)
    all_valid &= is_valid

    status = "OK" if is_valid else "MISMATCH"
    print(f"  {stakeholder:<25s}: {total:6.2f}  [{status}]")


Check weights per stakeholder:
  Nature                   :   1.00  [OK]
  Companies                :   1.00  [OK]
  Government               :   1.00  [OK]
  Citizents                :   1.00  [OK]


## Stakeholder Weights

Just as each stakeholder weighs their own criteria, the five stakeholder groups themselves must be weighted relative to one another before their individual preference scores can be combined into a single group score.

$$\sum_{i=1}^{5} w_i = 1$$

In this model, all stakeholder groups are initially weighted equally ($w_i = \tfrac{1}{5} = 0.2$), meaning no single stakeholder's preferences dominate the outcome. These weights can later be adjusted to reflect, for example, a decision-maker's judgement about whose interests should carry more weight.


In [73]:
# Set stakeholder weights
#               Nat    Comp  gov  cite
weights_eq =    [0.25, 0.25, 0.25, 0.25]  # equal weights for stakeholders
weights_dom =   [0.15, 0.25, 0.4, 0.2]  # city council dominant weight

stakeholder_weights = weights_eq 

assert np.isclose(sum(stakeholder_weights), 1), f"Weights must sum to 1, got {sum(stakeholder_weights)}"


## Aggregating the Preference Scores

The final group preference score per alternative is calculated using the `a_fine_aggregator`, in **two levels**:

1. **Level 1 — criteria → stakeholder score.** For each stakeholder, their ratings of the four alternatives on each criterion are combined using that stakeholder's own criteria weights. This produces one aggregated preference score per stakeholder per alternative.
2. **Level 2 — stakeholders → group score.** The resulting stakeholder scores (one row per stakeholder, one column per alternative) are combined a second time, now using the `stakeholder_weights` defined above, to produce a single final preference score per alternative.


The argument `scores_range=(-0.0, -100.0)` specifies that the aggregated preference scores are normalized to a range from **0** (least preferred) to **100** (most preferred). The negative range is intentional, as the affine aggregator is formulated for minimization problems and therefore operates on negative preference values.

In [74]:
print(ratings[alternatives].dtypes)

for col in alternatives:
    print(f"\n{col}:")
    print(ratings[col].unique())

Tunnel                   int64
Don't widen river        int64
Truss bridge             int64
Bridge somewhere else    int64
Werkspoorbridge          int64
dtype: object

Tunnel:
[ 2  9  6 10  8  5  3  7  4]

Don't widen river:
[10  5  3  4  2  8  1  9]

Truss bridge:
[ 5  8  7  6 10  9]

Bridge somewhere else:
[5 8 7 9 4 6 3]

Werkspoorbridge:
[ 5  8  7  6 10  9]


In [75]:
# Calculate the aggregated scores for each alternative using the a-fine-aggregator
# --- Level 1: aggregate criteria ratings -> one score per stakeholder per alternative ---
stakeholder_scores = {}

for stakeholder in stakeholders:
    stakeholder_data = ratings.loc[ratings["Stakeholder"] == stakeholder]
    criteria_weights = stakeholder_data["Criteria_weight"].to_numpy()
    p = stakeholder_data[alternatives].to_numpy()  # shape: n_criteria x n_alternatives
    stakeholder_scores[stakeholder] = a_fine_aggregator(criteria_weights, p, scores_range=(-0.0, -100.0))

# Collect into matrix: rows = stakeholders, columns = alternatives (order matches `alternatives`)
stakeholder_score_matrix = np.array([stakeholder_scores[s] for s in stakeholders])

print("Individual stakeholder aggregated scores:")
display(pd.DataFrame(stakeholder_score_matrix, index=stakeholders, columns=alternatives))

# --- Level 2: aggregate stakeholder scores -> final preference score per alternative ---
final_scores = a_fine_aggregator(stakeholder_weights, stakeholder_score_matrix, scores_range=(-0.0, -100.0))

results = (
    pd.DataFrame(final_scores, index=alternatives, columns=["Preference score"])
    .round(2)
    .sort_values("Preference score", ascending=False)
)

print("Final aggregated preference scores per alternative:")
display(results)


Individual stakeholder aggregated scores:


,Tunnel,Don't widen river,Truss bridge,Bridge somewhere else,Werkspoorbridge
Nature,0.00,100.00,53.80,34.30,75.29
Companies,96.19,0.00,95.30,100.00,95.30
Government,40.14,0.00,96.51,71.37,100.00
Citizents,58.64,100.00,84.38,0.00,98.38


Final aggregated preference scores per alternative:


,Preference score
Werkspoorbridge,100.00
Truss bridge,77.34
Don't widen river,9.87
Bridge somewhere else,7.38
Tunnel,0.00


## Interpreting the Results

The `results` table ranks the four alternatives by their final group preference score, from most to least preferred across all five stakeholder groups combined. Because the underlying scores are normalized before being combined, the results reflect each alternative's *relative* standing among the options considered — not an absolute measure of quality.

It's worth revisiting the `stakeholder_weights` (and, further upstream, each stakeholder's `Criteria_Weight` values in the CSV) to see how sensitive the final ranking is to these assumptions — a common next step in an MCDA is a simple sensitivity or scenario analysis.
